# Train Models

Цель ноутбука - обучить модели на лучших датасетах из `data/processed`, подобрать гиперпараметры и подготовить код логирования результатов в MLflow.

Используем два готовых набора признаков:

- `data/processed/linear/train.csv` - для логистической регрессии;
- `data/processed/tree/train.csv` - для дерева решений, XGBoost, CatBoost и случайного леса.

MLflow-блоки ниже отключены флагом `ENABLE_MLFLOW = False`, чтобы ноутбук можно было выполнять без поднятого MLflow tracking server. Когда сервер и пакет `mlflow` будут доступны, достаточно переключить флаг и указать `MLFLOW_TRACKING_URI`.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

src_path = PROJECT_ROOT / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from credit_risk_scoring.modeling.train_models import (
    MODEL_CONFIGS,
    RANDOM_STATE,
    load_training_data,
    save_training_artifacts,
    train_tuned_model,
)

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 240)

LINEAR_TRAIN_PATH = PROJECT_ROOT / "data" / "processed" / "linear" / "train.csv"
TREE_TRAIN_PATH = PROJECT_ROOT / "data" / "processed" / "tree" / "train.csv"

MODEL_DATASETS = {
    "logistic_regression": LINEAR_TRAIN_PATH,
    "decision_tree": TREE_TRAIN_PATH,
    "xgboost": TREE_TRAIN_PATH,
    "catboost": TREE_TRAIN_PATH,
    "random_forest": TREE_TRAIN_PATH,
}

PROJECT_ROOT

## Проверка входных данных

Перед обучением фиксируем размеры датасетов, число признаков и долю дефолта. Это полезно для контроля: если после изменения feature engineering эти числа резко изменятся, результаты обучения тоже могут заметно уехать.

In [ ]:
dataset_overview = []

for dataset_name, path in {
    "linear": LINEAR_TRAIN_PATH,
    "tree": TREE_TRAIN_PATH,
}.items():
    X, y = load_training_data(path)
    dataset_overview.append(
        {
            "dataset": dataset_name,
            "path": str(path.relative_to(PROJECT_ROOT)),
            "rows": X.shape[0],
            "features": X.shape[1],
            "target_rate": y.mean(),
        }
    )

dataset_overview = pd.DataFrame(dataset_overview)
dataset_overview

## Настройки подбора

Критерий оптимизации - `roc_auc`, потому что задача кредитного скоринга вероятностная и классы несбалансированы. Precision, recall, F1 и average precision считаются отдельно на валидационном holdout-сплите.

Для логистической регрессии и дерева решений используем полный grid search. Для XGBoost, CatBoost и случайного леса используем random search: сетки там многомерные, и полный перебор быстро становится слишком дорогим.

In [ ]:
SEARCH_PLAN = {
    "logistic_regression": {
        "search_strategy": "grid",
        "cv": 5,
        "n_jobs": -1,
    },
    "decision_tree": {
        "search_strategy": "grid",
        "cv": 5,
        "n_jobs": -1,
    },
    "xgboost": {
        "search_strategy": "random",
        "n_iter": 30,
        "cv": 5,
        "n_jobs": 1,
    },
    "catboost": {
        "search_strategy": "random",
        "n_iter": 25,
        "cv": 5,
        "n_jobs": 1,
    },
    "random_forest": {
        "search_strategy": "random",
        "n_iter": 20,
        "cv": 5,
        "n_jobs": 1,
    },
}

EXPERIMENTS = {}


def summarize_result(model_type: str, result: dict) -> dict:
    metrics = result["metrics"]
    validation = metrics["validation"]
    train = metrics["train"]

    return {
        "model_type": model_type,
        "dataset": metrics["dataset_family"],
        "features": metrics["features"],
        "target_rate": metrics["target_rate"],
        "best_cv_roc_auc": metrics["best_cv_score"],
        "train_roc_auc": train["roc_auc"],
        "valid_roc_auc": validation["roc_auc"],
        "valid_average_precision": validation["average_precision"],
        "valid_precision": validation["precision"],
        "valid_recall": validation["recall"],
        "valid_f1": validation["f1"],
        "best_params": metrics["best_params"],
    }


def run_model_experiment(model_type: str) -> dict:
    plan = SEARCH_PLAN[model_type]
    train_path = MODEL_DATASETS[model_type]

    result = train_tuned_model(
        train_path=train_path,
        model_type=model_type,
        search_strategy=plan["search_strategy"],
        scoring="roc_auc",
        cv=plan.get("cv", 5),
        n_jobs=plan.get("n_jobs", -1),
        n_iter=plan.get("n_iter", 25),
        test_size=0.2,
        random_state=RANDOM_STATE,
    )
    EXPERIMENTS[model_type] = result

    summary = pd.DataFrame([summarize_result(model_type, result)])
    display(summary.drop(columns=["best_params"]))
    display(pd.Series(result["metrics"]["best_params"], name="best_params"))
    return result

## MLflow logging helper

Это рабочий код логирования, но он не запускается, пока `ENABLE_MLFLOW = False`. Такой режим удобен для локального ноутбука: подбор моделей можно выполнить сейчас, а логирование включить позже, когда будет выбран tracking URI.

In [ ]:
ENABLE_MLFLOW = False
MLFLOW_TRACKING_URI = "http://127.0.0.1:5000"
MLFLOW_EXPERIMENT_NAME = "give-me-some-credit-model-training"


def log_result_to_mlflow(model_type: str, result: dict) -> None:
    if not ENABLE_MLFLOW:
        print(f"MLflow logging is disabled for {model_type}.")
        return

    import mlflow
    import mlflow.catboost
    import mlflow.sklearn

    metrics = result["metrics"]
    model = result["model"]

    mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
    mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

    with mlflow.start_run(run_name=model_type):
        mlflow.log_param("model_type", model_type)
        mlflow.log_param("dataset_family", metrics["dataset_family"])
        mlflow.log_param("train_path", metrics["train_path"])
        mlflow.log_param("rows", metrics["rows"])
        mlflow.log_param("features", metrics["features"])
        mlflow.log_param("search_strategy", metrics["search_strategy"])
        mlflow.log_param("scoring", metrics["scoring"])
        mlflow.log_param("cv", metrics["cv"])
        mlflow.log_params(metrics["best_params"])

        mlflow.log_metric("target_rate", metrics["target_rate"])
        mlflow.log_metric("best_cv_score", metrics["best_cv_score"])

        for split_name in ["train", "validation"]:
            for metric_name, metric_value in metrics[split_name].items():
                mlflow.log_metric(f"{split_name}_{metric_name}", metric_value)

        if model_type == "catboost":
            mlflow.catboost.log_model(model, artifact_path="model")
        else:
            mlflow.sklearn.log_model(model, artifact_path="model")

## 1. Logistic Regression

Логистическую регрессию обучаем на `linear`-датасете: он содержит признаки, выбранные специально для линейной модели, включая логарифмы и dummy-признаки после дискретизации.

In [ ]:
logistic_regression_result = run_model_experiment("logistic_regression")
log_result_to_mlflow("logistic_regression", logistic_regression_result)

## 2. Decision Tree

Дерево решений обучаем на `tree`-датасете. В нем оставлены исходные числовые признаки и маркеры риска, потому что дерево само ищет нелинейные разбиения.

In [ ]:
decision_tree_result = run_model_experiment("decision_tree")
log_result_to_mlflow("decision_tree", decision_tree_result)

## 3. XGBoost

XGBoost также использует `tree`-датасет. В обучающем модуле для него дополнительно задается `scale_pos_weight`, рассчитанный по доле классов на train-сплите.

In [ ]:
xgboost_result = run_model_experiment("xgboost")
log_result_to_mlflow("xgboost", xgboost_result)

## 4. CatBoost

CatBoost обучаем на том же `tree`-датасете. Для дисбаланса классов используется `auto_class_weights="Balanced"`, а служебные файлы CatBoost отключены через `allow_writing_files=False`.

In [ ]:
catboost_result = run_model_experiment("catboost")
log_result_to_mlflow("catboost", catboost_result)

## 5. Random Forest

Случайный лес - ансамблевая модель на `tree`-датасете. Для дисбаланса используется `class_weight="balanced_subsample"`.

In [ ]:
random_forest_result = run_model_experiment("random_forest")
log_result_to_mlflow("random_forest", random_forest_result)

## Сравнение оптимальных моделей

После выполнения пяти блоков ниже появляется общая таблица. Сортировка идет по ROC-AUC на валидации, затем по recall и precision.

In [ ]:
comparison = pd.DataFrame(
    [summarize_result(model_type, result) for model_type, result in EXPERIMENTS.items()]
).sort_values(
    by=["valid_roc_auc", "valid_recall", "valid_precision"],
    ascending=[False, False, False],
).reset_index(drop=True)

comparison

## Зафиксированные лучшие результаты

После запуска подбора лучшие параметры и качество сохранены в `reports/modeling/best_model_results.md`, `reports/modeling/model_selection_results.json` и `reports/modeling/model_selection_results.csv`.

| Model | Best CV ROC-AUC | Validation ROC-AUC | Best params |
|---|---:|---:|---|
| logistic_regression | 0.861040 | 0.859239 | `model__C=5.0`, `model__solver=liblinear` |
| decision_tree | 0.854057 | 0.848535 | `criterion=entropy`, `max_depth=7`, `min_samples_leaf=200`, `min_samples_split=50` |
| xgboost | 0.866852 | 0.863794 | `model__n_estimators=400`, `model__max_depth=4`, `model__learning_rate=0.03`, `model__subsample=1.0`, `model__colsample_bytree=0.8`, `model__reg_lambda=10.0` |
| catboost | 0.867065 | 0.863498 | `iterations=700`, `depth=6`, `learning_rate=0.03`, `l2_leaf_reg=10.0` |
| random_forest | 0.864676 | 0.861058 | `n_estimators=500`, `max_depth=12`, `min_samples_leaf=20`, `max_features=sqrt` |

По holdout ROC-AUC лучшая модель - `xgboost`: `0.863794`. По CV ROC-AUC лучший результат у `catboost`: `0.867065`.

## Сохранение артефактов

Основной воспроизводимый путь сохранения моделей находится в DVC/Airflow-пайплайне. Если нужно сохранить результаты прямо из ноутбука, переключите `SAVE_ARTIFACTS = True`.

In [ ]:
SAVE_ARTIFACTS = False

if SAVE_ARTIFACTS:
    for model_type, result in EXPERIMENTS.items():
        save_training_artifacts(
            result,
            model_output_path=PROJECT_ROOT / "models" / f"{model_type}.joblib",
            metrics_output_path=PROJECT_ROOT / "reports" / "modeling" / f"{model_type}_metrics.json",
            params_output_path=PROJECT_ROOT / "reports" / "modeling" / f"{model_type}_best_params.json",
            cv_results_output_path=PROJECT_ROOT / "reports" / "modeling" / f"{model_type}_cv_results.csv",
        )

## Воспроизводимый запуск вне ноутбука

Для production-like запуска добавлены DVC-стадии и Airflow DAG:

```bash
dvc repro train-logistic-regression
dvc repro train-decision-tree
dvc repro train-xgboost
dvc repro train-catboost
dvc repro train-random-forest
```

Airflow DAG находится в `airflow/dags/train_optimal_models.py` и запускает эти стадии после пересборки feature-engineering датасетов.